In [ ]:
# ── 14a. Interactive prediction map ───────────────────────────────────────────
from IPython.display import IFrame

pred_map_path = os.path.join(maps_dir, "jam_predictions_map.html")
if os.path.isfile(pred_map_path):
    display(IFrame(pred_map_path, width=900, height=500))
else:
    print("Prediction map not found. Run python main.py first.")

# ── 14b. Interactive uncertainty map ─────────────────────────────────────────
unc_map_path = os.path.join(maps_dir, "jam_uncertainty_map.html")
if os.path.isfile(unc_map_path):
    print("\nUncertainty Map (CI width):")
    display(IFrame(unc_map_path, width=900, height=500))
else:
    print("Uncertainty map not found. Run python main.py first.")

## 14. Interactive Maps

In [ ]:
# ── 13. Sex-disaggregated analysis ────────────────────────────────────────────
sex_path = os.path.join(os.path.dirname(eval_dir), "..", "interim", "jam_targets_sex.csv")
# Normalize the path
sex_path = os.path.normpath(sex_path)
if not os.path.isfile(sex_path):
    sex_path = os.path.join(os.path.dirname(eval_dir), "..", "interim", "jam_targets_sex.csv")

if os.path.isfile(sex_path):
    sex_df = pd.read_csv(sex_path)
    print("Sex-Disaggregated Poverty Targets (Jamaica 2022)")
    print("=" * 60)
    display(sex_df.round(2))

    # Bar chart: male vs female by zone
    if len(sex_df) > 0:
        pivot = sex_df.pivot_table(
            index="subregion", columns="sex", values="moderate_prevalence"
        )
        fig, ax = plt.subplots(figsize=(10, 5))
        pivot.plot(kind="bar", ax=ax, color=["#d73027", "#4a86c8"])
        ax.set_ylabel("Moderate Poverty Prevalence (%)")
        ax.set_title("Sex Gap in Child Poverty by Zone")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.show()
else:
    print(f"Sex-disaggregated targets not found at {sex_path}.")

## 13. Sex-Disaggregated Poverty Analysis

In [ ]:
# ── 12. Depth metrics comparison ──────────────────────────────────────────────
depth_cols = [c for c in predictions.columns if c.endswith("_moderate_depth")]
if depth_cols and "moderate_depth" in predictions.columns:
    depth_data = predictions[predictions["moderate_depth"].notna()].copy()
    print(f"Depth metric columns found: {depth_cols}")
    print(f"\nDepth predictions summary ({len(depth_data)} cells):")
    display(depth_data[["subregion", "moderate_depth"] + depth_cols].groupby("subregion").mean().round(4))
else:
    print("No depth metric columns found in predictions.")

## 12. Depth Metrics Comparison

In [ ]:
# ── 11. Significance Tests ────────────────────────────────────────────────────
sig_path = os.path.join(eval_dir, "significance_tests.csv")
if os.path.isfile(sig_path):
    sig_df = pd.read_csv(sig_path)
    print("Paired Significance Tests: ML methods vs RWI baseline")
    print("=" * 60)
    display(sig_df.round(6))
else:
    print("Significance tests not found. Run python main.py first.")

## 11. Statistical Significance Tests

In [ ]:
# ── 10. LOZO Results ──────────────────────────────────────────────────────────
lozo_path = os.path.join(eval_dir, "lozo_evaluation.csv")
if os.path.isfile(lozo_path):
    lozo_df = pd.read_csv(lozo_path)
    print("Leave-One-Zone-Out Cross-Validation Results")
    print("=" * 60)
    display(lozo_df.round(4))

    # Pivot for cleaner view
    if len(lozo_df) > 0:
        pivot = lozo_df.pivot_table(index="zone", columns="method", values="abs_error")
        fig, ax = plt.subplots(figsize=(10, 4))
        pivot.plot(kind="bar", ax=ax)
        ax.set_ylabel("Absolute Error (pp)")
        ax.set_title("LOZO: Held-Out Zone Prediction Error by Method")
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()
else:
    print(f"LOZO results not found. Run python main.py first.")

## 10. Leave-One-Zone-Out Cross-Validation

In [ ]:
# ── 9. SHAP bar chart ──────────────────────────────────────────────────────────
import os

shap_path = os.path.join(eval_dir, "shap_summary.csv")
if os.path.isfile(shap_path):
    shap_df = pd.read_csv(shap_path)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(shap_df["feature"], shap_df["mean_abs_shap"], color="#4a86c8")
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_title("GBM SHAP Feature Importance")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print(f"SHAP summary not found at {shap_path}. Run pipeline with shap installed.")

## 9. SHAP Feature Importance (GBM)

# UNICEF × RBC Borealis AI — Jamaica Child Deprivation Analysis
## Spatial Disaggregation Pipeline: Results, Evaluation & Visualisation

**Project**: AI-Powered Reconstruction of Fine-Scale Child Deprivation for Disaster Forecasting  
**Country**: Jamaica (pilot)  
**Survey**: MICS 2022 — Moderate & Severe Child Poverty Prevalence  
**Program**: RBC Borealis AI — Let's SOLVE It, Spring 2026

---

> ⚠️ **Research disclaimer**: All predictions are spatial disaggregations consistent with official totals — they are **not official statistics**. Spatial variation within zones is inferred from proxy signals (RWI, population, settlement type, travel time), not directly observed.

### Notebook Structure

| Section | Contents |
|---|---|
| 1 | Exploratory Data Analysis — grid, features, spatial coverage |
| 2 | Target Data — official zone-level poverty prevalences |
| 3 | Prediction Comparison — all four methods side-by-side |
| 4 | Spatial Maps — scatter maps of predictions across Jamaica |
| 5 | Administrative Reconciliation — verification that zone totals are preserved |
| 6 | Uncertainty Bands — Ridge bootstrap confidence intervals |
| 7 | Feature Importance — Ridge coefficients and GBM feature importances |
| 8 | Evaluation Summary — comparative metrics table + caveats |

In [ ]:
import sys
import os

# Add project root to path so src modules are importable
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec
from scipy import stats

# ── Plot style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi": 130,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

# ── Colour palette ──────────────────────────────────────────────────────────
ZONE_COLORS   = {"Urban": "#4C72B0", "Rural": "#DD8452", "Kingston Metropolitan Area (KMA)": "#55A868", "Unknown": "#AAAAAA"}
METHOD_COLORS = {"uniform": "#999999", "rwi": "#E377C2", "ridge": "#2CA02C", "gbm": "#D62728"}
METHOD_LABELS = {"uniform": "Uniform", "rwi": "RWI redistribution", "ridge": "Ridge regression", "gbm": "Gradient Boosting"}

print("✓ Setup complete.")

In [ ]:
# ── Load all pipeline outputs ───────────────────────────────────────────────
modeling_table = pd.read_parquet(os.path.join(PROJECT_ROOT, "data/interim/jam_modeling_table.parquet"))
targets        = pd.read_csv(os.path.join(PROJECT_ROOT, "data/interim/jam_targets.csv"))
predictions    = pd.read_parquet(os.path.join(PROJECT_ROOT, "data/outputs/tables/jam_predictions.parquet"))

# Load evaluation files
eval_dir = os.path.join(PROJECT_ROOT, "data/outputs/eval")
eval_summary = pd.read_csv(os.path.join(eval_dir, "evaluation_summary.csv"), index_col=0)

# Load admin detail per method (if present)
admin_details = {}
for method in ["uniform_moderate", "rwi_moderate", "ridge_moderate", "gbm_moderate"]:
    fpath = os.path.join(eval_dir, f"admin_detail_{method}.csv")
    if os.path.exists(fpath):
        admin_details[method] = pd.read_csv(fpath)

# Load GADM parish boundaries for map overlays
gadm_path = os.path.join(PROJECT_ROOT, "Data/Geospatial/gadm41_JAM.gpkg")
parishes = gpd.read_file(gadm_path, layer="ADM_ADM_1")

# ── Quick sanity check ──────────────────────────────────────────────────────
print(f"Modeling table : {modeling_table.shape[0]:,} rows × {modeling_table.shape[1]} columns")
print(f"Predictions    : {predictions.shape[0]:,} rows × {predictions.shape[1]} columns")
print(f"In modeling sample: {modeling_table['in_modeling_sample'].sum():,} / {len(modeling_table):,}")
print(f"\nZone breakdown:")
print(modeling_table.groupby("subregion").size().rename("cells").to_string())
print(f"\nTargets (Jamaica 2022 MICS):")
print(targets[["subregion", "moderate_prevalence", "severe_prevalence"]].to_string(index=False))
print(f"\nPrediction columns: {[c for c in predictions.columns if any(c.startswith(m) for m in ['uniform','rwi','ridge','gbm'])]}")

---
## Section 1 — Exploratory Data Analysis

Understanding the spatial coverage of the base grid and the distribution of proxy features.

In [ ]:
# ── 1a. Grid spatial coverage coloured by subregion ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: All grid points coloured by subregion
ax = axes[0]
for zone, grp in modeling_table.groupby("subregion"):
    ax.scatter(grp["longitude"], grp["latitude"],
               c=ZONE_COLORS.get(zone, "#AAAAAA"), s=6, alpha=0.7,
               label=zone, zorder=3)
parishes.boundary.plot(ax=ax, color="#333333", linewidth=0.5, zorder=2)
ax.set_title("Base Grid — Subregion Assignment\n(1,745 RWI points, ~2.3 km spacing)")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.legend(loc="lower left", markerscale=2, framealpha=0.9, fontsize=8)
ax.set_aspect("equal")

# Right: RWI distribution across the island
ax2 = axes[1]
rwi_norm = plt.Normalize(modeling_table["rwi"].min(), modeling_table["rwi"].max())
sc = ax2.scatter(modeling_table["longitude"], modeling_table["latitude"],
                 c=modeling_table["rwi"], cmap="RdYlGn", norm=rwi_norm, s=6, alpha=0.8)
parishes.boundary.plot(ax=ax2, color="#333333", linewidth=0.5)
plt.colorbar(sc, ax=ax2, label="Relative Wealth Index (RWI)", shrink=0.85)
ax2.set_title("Relative Wealth Index\n(green = wealthier, red = poorer)")
ax2.set_xlabel("Longitude")
ax2.set_aspect("equal")

plt.tight_layout()
plt.suptitle("Figure 1 — Base Grid: Spatial Coverage & Wealth Index", fontsize=13, fontweight="bold", y=1.02)
plt.show()

print(f"\nRWI summary:\n{modeling_table['rwi'].describe().round(3).to_string()}")

In [ ]:
# ── 1b. Feature distributions by subregion ───────────────────────────────────
features = ["rwi", "population", "travel_time_cities", "travel_time_50k"]
feature_labels = {
    "rwi": "Relative Wealth Index",
    "population": "Population (WorldPop 2030)",
    "travel_time_cities": "Travel Time to Cities (min)",
    "travel_time_50k": "Travel Time to 50k Centre (min)",
}

df_model = modeling_table[modeling_table["in_modeling_sample"] == True].copy()
zones_ordered = ["Urban", "Kingston Metropolitan Area (KMA)", "Rural"]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for i, feat in enumerate(features):
    ax = axes[i]
    data_by_zone = [df_model[df_model["subregion"] == z][feat].dropna().values
                    for z in zones_ordered]
    short_labels = ["Urban", "KMA", "Rural"]
    colors = [ZONE_COLORS[z] for z in zones_ordered]

    bp = ax.boxplot(data_by_zone, patch_artist=True, notch=False,
                    medianprops=dict(color="black", linewidth=2))
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    ax.set_xticklabels(short_labels)
    ax.set_title(feature_labels[feat])
    ax.set_ylabel("Value")

plt.suptitle("Figure 2 — Proxy Feature Distributions by Subregion", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# ── Correlation matrix of features ──────────────────────────────────────────
print("\nFeature correlation matrix (Pearson r):")
corr_cols = ["rwi", "population", "smod_class", "travel_time_cities", "travel_time_50k", "is_urban"]
print(df_model[corr_cols].corr().round(3).to_string())

---
## Section 2 — Target Data

Official Jamaica 2022 MICS poverty prevalences — the only supervision signal available. These three zone-level values are the **ground truth** that all predictions are reconciled to.

In [ ]:
# ── Target prevalence bar chart ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
zone_order = ["Urban", "Kingston Metropolitan Area (KMA)", "Rural"]
short = ["Urban", "KMA", "Rural"]
tgt = targets.set_index("subregion")

for ax, col, title in zip(
    axes,
    ["moderate_prevalence", "severe_prevalence"],
    ["Moderate Child Poverty (%)", "Severe Child Poverty (%)"]
):
    vals = [tgt.loc[z, col] if z in tgt.index else np.nan for z in zone_order]
    colors = [ZONE_COLORS[z] for z in zone_order]
    bars = ax.bar(short, vals, color=colors, edgecolor="white", linewidth=0.8, width=0.55)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f"{val:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")
    ax.set_ylim(0, max(vals) * 1.25)
    ax.set_ylabel("Prevalence (%)")
    ax.set_title(title)
    ax.axhline(y=np.mean(vals), color="grey", linestyle="--", linewidth=1, alpha=0.6, label=f"Mean: {np.mean(vals):.1f}%")
    ax.legend(fontsize=8)

plt.suptitle("Figure 3 — Official Zone-Level Poverty Targets (Jamaica 2022 MICS)\n"
             "Source: UNICEF MICS — NOT for direct comparison with model outputs",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nKey observation:")
print("  • Rural and KMA have near-identical moderate poverty (34.6% vs 34.6%)")
print("  • Urban is notably lower (22.9%) — KMA cities included in Urban, so KMA/Rural are genuinely more deprived")
print("  • Severe poverty: KMA higher than Rural (18.3% vs 13.8%) — KMA urban poverty is concentrated")
print(f"\n  Cells per zone: {df_model.groupby('subregion').size().to_dict()}")

---
## Section 3 — Prediction Comparison

Comparing the four methods side-by-side. Because all methods are reconciled to the same zone totals, differences in prediction values reflect **within-zone spatial variation** only.

In [ ]:
# ── 3a. Prediction distributions (violin plots per zone per method) ──────────
pred_methods = [("uniform_moderate", "Uniform"), ("rwi_moderate", "RWI"),
                ("ridge_moderate", "Ridge"), ("gbm_moderate", "GBM")]
pred_methods = [(col, lbl) for col, lbl in pred_methods if col in predictions.columns]

df_pred = predictions[predictions["moderate_prevalence"].notna()].copy()

fig, axes = plt.subplots(1, len(pred_methods), figsize=(14, 5), sharey=True)
if len(pred_methods) == 1:
    axes = [axes]

for ax, (col, lbl) in zip(axes, pred_methods):
    data_by_zone = [df_pred[df_pred["subregion"] == z][col].dropna().values
                    for z in zones_ordered]
    short_labels = ["Urban", "KMA", "Rural"]
    colors = [ZONE_COLORS[z] for z in zones_ordered]

    parts = ax.violinplot(data_by_zone, positions=range(len(zones_ordered)),
                          showmedians=True, showextrema=True)
    for pc, color in zip(parts["bodies"], colors):
        pc.set_facecolor(color)
        pc.set_alpha(0.6)
    parts["cmedians"].set_color("black")
    parts["cbars"].set_color("black")
    parts["cmins"].set_color("black")
    parts["cmaxes"].set_color("black")

    # Overlay official target as horizontal dashed line per zone
    for j, zone in enumerate(zones_ordered):
        target_val = tgt.loc[zone, "moderate_prevalence"] if zone in tgt.index else None
        if target_val:
            ax.hlines(target_val, j - 0.4, j + 0.4, colors="red",
                      linewidths=1.5, linestyles="--", alpha=0.9)

    ax.set_xticks(range(len(zones_ordered)))
    ax.set_xticklabels(short_labels)
    ax.set_title(f"{lbl}", fontsize=11, fontweight="bold")
    if ax == axes[0]:
        ax.set_ylabel("Moderate Poverty Prevalence (%)")

# Legend
legend_els = [Patch(facecolor="red", alpha=0.8, label="Official zone target")]
axes[-1].legend(handles=legend_els, loc="upper right", fontsize=8)

plt.suptitle("Figure 4 — Prediction Distributions Within Zones (moderate poverty)\n"
             "Red dashed line = official target. Uniform = zero within-zone spread by design.",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── 3b. Pairwise scatter: RWI-based redistribution vs other methods ──────────
compare_cols = [c for c in ["uniform_moderate", "ridge_moderate", "gbm_moderate"] if c in df_pred.columns]
n_compare = len(compare_cols)

if n_compare > 0:
    fig, axes = plt.subplots(1, n_compare, figsize=(5 * n_compare, 5))
    if n_compare == 1:
        axes = [axes]

    for ax, col in zip(axes, compare_cols):
        method_key = col.replace("_moderate", "")
        for zone in zones_ordered:
            zm = df_pred["subregion"] == zone
            ax.scatter(df_pred.loc[zm, "rwi_moderate"],
                       df_pred.loc[zm, col],
                       c=ZONE_COLORS[zone], s=10, alpha=0.5,
                       label=zone.replace("Kingston Metropolitan Area ", ""))

        # Identity line
        lims = [min(df_pred[["rwi_moderate", col]].min()), max(df_pred[["rwi_moderate", col]].max())]
        ax.plot(lims, lims, "k--", alpha=0.4, linewidth=1, label="y = x")

        r, p = stats.pearsonr(
            df_pred["rwi_moderate"].dropna(),
            df_pred[col].reindex(df_pred["rwi_moderate"].dropna().index).fillna(df_pred[col].mean())
        )
        ax.set_xlabel("RWI redistribution (%)")
        ax.set_ylabel(f"{METHOD_LABELS.get(method_key, method_key)} (%)")
        ax.set_title(f"RWI vs {METHOD_LABELS.get(method_key, method_key)}\nr = {r:.3f}")
        ax.legend(markerscale=2, fontsize=7, loc="upper left")

    plt.suptitle("Figure 5 — RWI Redistribution vs Other Methods (moderate poverty, per-cell)",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

---
## Section 4 — Spatial Maps

Each panel shows the predicted moderate poverty prevalence across Jamaica for one method. The colour scale is shared across all panels for direct comparison.

In [ ]:
# ── 4. Spatial maps — one per prediction method ──────────────────────────────
map_methods = [("uniform_moderate", "Uniform"),
               ("rwi_moderate", "RWI Redistribution"),
               ("ridge_moderate", "Ridge Regression"),
               ("gbm_moderate", "Gradient Boosting")]
map_methods = [(col, lbl) for col, lbl in map_methods if col in df_pred.columns]

# Shared colour scale across all panels
all_preds = pd.concat([df_pred[col] for col, _ in map_methods]).dropna()
vmin, vmax = all_preds.quantile(0.02), all_preds.quantile(0.98)
cmap = plt.cm.YlOrRd

n_maps = len(map_methods)
ncols = 2
nrows = (n_maps + 1) // 2

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 5 * nrows))
axes = axes.flatten()

for ax, (col, lbl) in zip(axes, map_methods):
    parishes.boundary.plot(ax=ax, color="#555555", linewidth=0.4)
    sc = ax.scatter(df_pred["longitude"], df_pred["latitude"],
                    c=df_pred[col], cmap=cmap, vmin=vmin, vmax=vmax,
                    s=6, alpha=0.85)
    ax.set_title(lbl, fontsize=11, fontweight="bold")
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.set_aspect("equal")
    plt.colorbar(sc, ax=ax, label="Moderate poverty (%)", shrink=0.85, pad=0.02)

# Hide unused axes
for ax in axes[n_maps:]:
    ax.set_visible(False)

plt.suptitle("Figure 6 — Spatial Maps of Predicted Moderate Child Poverty\n"
             "(shared colour scale | red = higher deprivation | yellow = lower deprivation)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print("Note: Uniform baseline shows sharp zone boundaries (no within-zone variation).")
print("RWI, Ridge, and GBM produce smooth gradients informed by proxy features.")

---
## Section 5 — Administrative Reconciliation Verification

All methods apply hard reconciliation: the population-weighted mean within each zone exactly matches the official target. This section verifies that constraint visually.

In [ ]:
# ── 5. Reconciliation check — achieved zone means vs official targets ─────────
pop = df_pred["population"].values.astype(float)
pop = np.where(np.isnan(pop) | (pop < 0), 0.0, pop)
rec_rows = []

for col, lbl in [("uniform_moderate","Uniform"), ("rwi_moderate","RWI"),
                 ("ridge_moderate","Ridge"), ("gbm_moderate","GBM")]:
    if col not in df_pred.columns:
        continue
    for zone in zones_ordered:
        zm = df_pred["subregion"] == zone
        target_val = tgt.loc[zone, "moderate_prevalence"] if zone in tgt.index else np.nan
        z_pop = pop[zm.values]
        z_pred = df_pred.loc[zm, col].values
        achieved = np.average(z_pred, weights=z_pop) if z_pop.sum() > 0 else z_pred.mean()
        rec_rows.append({"method": lbl, "zone": zone.replace("Kingston Metropolitan Area ",""),
                         "target": target_val, "achieved": achieved,
                         "error_pp": abs(achieved - target_val)})

rec_df = pd.DataFrame(rec_rows)

# Plot
fig, ax = plt.subplots(figsize=(11, 4.5))
x = np.arange(len(zones_ordered))
width = 0.18
methods_ordered = rec_df["method"].unique()
offsets = np.linspace(-(len(methods_ordered)-1)/2 * width, (len(methods_ordered)-1)/2 * width, len(methods_ordered))
short_zone = ["Urban", "KMA", "Rural"]
method_color_map = {"Uniform": METHOD_COLORS["uniform"], "RWI": METHOD_COLORS["rwi"],
                    "Ridge": METHOD_COLORS["ridge"], "GBM": METHOD_COLORS["gbm"]}

for i, method in enumerate(methods_ordered):
    vals = [rec_df[(rec_df["method"]==method) & (rec_df["zone"]==z)]["achieved"].values[0]
            for z in short_zone]
    ax.bar(x + offsets[i], vals, width=width * 0.9,
           label=method, color=method_color_map.get(method, "#888888"), alpha=0.85)

# Official target as horizontal tick marks
for j, zone in enumerate(zones_ordered):
    t = tgt.loc[zone, "moderate_prevalence"] if zone in tgt.index else None
    if t:
        ax.hlines(t, j - 0.45, j + 0.45, colors="black", linewidths=2, linestyles="-",
                  label="Official target" if j == 0 else "")
        ax.text(j + 0.47, t + 0.15, f"{t:.1f}%", fontsize=8, va="bottom", ha="left")

ax.set_xticks(x)
ax.set_xticklabels(short_zone, fontsize=10)
ax.set_ylabel("Population-Weighted Mean Prevalence (%)")
ax.set_title("Figure 7 — Reconciliation: Achieved Zone Means vs Official Targets\n"
             "All methods must land exactly on the black line (official target)",
             fontsize=11, fontweight="bold")
ax.legend(loc="upper right", fontsize=9)
plt.tight_layout()
plt.show()

print("\nMax reconciliation error across all methods × zones:")
print(f"  {rec_df['error_pp'].max():.8f} percentage points (effectively machine-zero)")

---
## Section 6 — Uncertainty Bands (Ridge Bootstrap)

Bootstrap resampling (50 iterations) gives 5th–95th percentile intervals for the Ridge model predictions.

In [ ]:
# ── 6a. Uncertainty CI width distribution ────────────────────────────────────
has_ci = ("ridge_moderate_lower" in df_pred.columns and
          "ridge_moderate_upper" in df_pred.columns)

if has_ci:
    df_ci = df_pred[df_pred["ridge_moderate_lower"].notna()].copy()
    df_ci["ci_width"] = df_ci["ridge_moderate_upper"] - df_ci["ridge_moderate_lower"]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # Left: CI width histogram by zone
    ax = axes[0]
    for zone in zones_ordered:
        zm = df_ci["subregion"] == zone
        ax.hist(df_ci.loc[zm, "ci_width"], bins=30, alpha=0.55,
                color=ZONE_COLORS[zone], label=zone.replace("Kingston Metropolitan Area ", "KMA "),
                density=True, edgecolor="white", linewidth=0.4)
    ax.set_xlabel("90% CI Width (percentage points)")
    ax.set_ylabel("Density")
    ax.set_title("Ridge Bootstrap CI Width Distribution")
    ax.legend(fontsize=8)

    # Right: Uncertainty map (CI width across Jamaica)
    ax2 = axes[1]
    parishes.boundary.plot(ax=ax2, color="#555555", linewidth=0.4)
    sc = ax2.scatter(df_ci["longitude"], df_ci["latitude"],
                     c=df_ci["ci_width"], cmap="PuBu", s=6, alpha=0.85)
    plt.colorbar(sc, ax=ax2, label="CI Width (pp)", shrink=0.85)
    ax2.set_title("Spatial Distribution of Uncertainty\n(wider = more uncertain)")
    ax2.set_xlabel("Longitude"); ax2.set_ylabel("Latitude")
    ax2.set_aspect("equal")

    plt.suptitle("Figure 8 — Ridge Bootstrap Prediction Uncertainty (90% CI)", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()

    print(f"\nCI width statistics:")
    print(df_ci.groupby("subregion")["ci_width"].describe()[["mean","std","min","max"]].round(2).to_string())
else:
    print("⚠ No uncertainty columns found (ridge_moderate_lower / upper) — re-run with Ridge model.")

In [ ]:
# ── 6b. Uncertainty band plot — sorted by Ridge prediction ───────────────────
if has_ci:
    # Sample 200 cells per zone for clarity
    sample_dfs = []
    for zone in zones_ordered:
        zm = df_ci[df_ci["subregion"] == zone].copy()
        zm = zm.sort_values("ridge_moderate").reset_index(drop=True)
        n_sample = min(200, len(zm))
        idx = np.linspace(0, len(zm)-1, n_sample, dtype=int)
        sample_dfs.append(zm.iloc[idx])

    fig, axes = plt.subplots(1, len(zones_ordered), figsize=(14, 4.5), sharey=True)
    for ax, (zone, sdf) in zip(axes, zip(zones_ordered, sample_dfs)):
        x_idx = np.arange(len(sdf))
        color = ZONE_COLORS[zone]
        ax.fill_between(x_idx, sdf["ridge_moderate_lower"], sdf["ridge_moderate_upper"],
                        alpha=0.3, color=color, label="90% CI")
        ax.plot(x_idx, sdf["ridge_moderate"], color=color, linewidth=1.2, label="Ridge pred.")
        # Official target
        t = tgt.loc[zone, "moderate_prevalence"] if zone in tgt.index else None
        if t:
            ax.axhline(t, color="red", linewidth=1.5, linestyle="--", label=f"Target: {t:.1f}%")
        ax.set_xlabel("Cells (sorted by prediction)")
        ax.set_title(zone.replace("Kingston Metropolitan Area ", "KMA "), fontsize=10)
        if ax == axes[0]:
            ax.set_ylabel("Moderate Poverty (%)")
        ax.legend(fontsize=7)

    plt.suptitle("Figure 9 — Ridge Bootstrap Uncertainty Bands per Zone\n"
                 "(sorted by prediction value — shaded region = 5th–95th percentile bootstrap)",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

---
## Section 7 — Feature Importance

### Ridge Regression Coefficients
Standardised coefficients show the direction and magnitude of each feature's effect on predicted deprivation (positive = more deprivation, negative = less).

### GBM Feature Importances
Gain-based importances from the gradient boosted trees (if run).

In [ ]:
# ── 7a. Ridge coefficients — refit model here to access coef table ────────────
from src.utils.config_loader import load_config
from src.models.ridge_model import RidgeDeprivationModel

cfg = load_config(os.path.join(PROJECT_ROOT, "config/config.yaml"))
feature_cols = cfg["modeling"]["features"]
df_train = modeling_table[modeling_table["in_modeling_sample"].fillna(False)].copy()
feature_mask = df_train[feature_cols].notna().all(axis=1)
X_train = df_train.loc[feature_mask, feature_cols].values.astype(float)
y_train = df_train.loc[feature_mask, "moderate_prevalence"].values.astype(float)

ridge_cfg = cfg["modeling"]["ridge"]
ridge = RidgeDeprivationModel(
    alpha_candidates=ridge_cfg["alpha_candidates"],
    cv_folds=ridge_cfg["cv_folds"],
    random_state=ridge_cfg["random_state"],
)
ridge.fit(X_train, y_train, feature_names=feature_cols)
coef_table = ridge.get_coef_table()

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
colors_bar = ["#D62728" if v > 0 else "#1F77B4" for v in coef_table["standardised_impact"]]
ax.barh(coef_table["feature"][::-1], coef_table["standardised_impact"][::-1],
        color=colors_bar[::-1], edgecolor="white", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Standardised Coefficient (impact on poverty prediction)")
ax.set_title("Figure 10 — Ridge Regression: Standardised Feature Coefficients\n"
             "Red = increases predicted deprivation | Blue = decreases predicted deprivation",
             fontsize=11, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nCoefficient table:")
print(coef_table.to_string(index=False))

In [ ]:
# ── 7c. GAM partial dependence curves ────────────────────────────────────────
try:
    from src.models.gam_model import GAMDeprivationModel

    gam_cfg = cfg["modeling"].get("gam", {})
    gam = GAMDeprivationModel(
        n_splines=gam_cfg.get("n_splines", 10),
        lam_candidates=gam_cfg.get("lam_candidates", [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]),
        max_iter=gam_cfg.get("max_iter", 100),
    )
    gam.fit(X_train, y_train, feature_names=feature_cols)
    pd_curves = gam.get_partial_dependence()

    n_feats = len(pd_curves)
    ncols = 3
    nrows = (n_feats + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4 * nrows))
    axes = axes.flatten() if nrows > 1 else [axes] if ncols == 1 else axes.flatten()

    for ax, (fname, pd_data) in zip(axes, pd_curves.items()):
        x_vals = pd_data["x"]
        y_vals = pd_data["y"]
        lo = pd_data["lower"]
        hi = pd_data["upper"]
        ax.plot(x_vals, y_vals, color="#2CA02C", linewidth=2)
        ax.fill_between(x_vals, lo, hi, alpha=0.2, color="#2CA02C")
        ax.axhline(0, color="grey", linewidth=0.8, linestyle="--", alpha=0.5)
        ax.set_title(fname, fontsize=9)
        ax.set_xlabel("Feature value")
        ax.set_ylabel("Partial effect on poverty")

    for ax in axes[n_feats:]:
        ax.set_visible(False)

    plt.suptitle("Figure 10b — GAM Partial Dependence Curves\n"
                 "(each panel shows the smooth effect of one feature, holding others constant)\n"
                 "Shaded band = 95% pointwise confidence interval",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

except ImportError:
    print("⚠ pygam not installed. Install with: pip install pygam")
except Exception as e:
    print(f"⚠ GAM partial dependence failed: {e}")

In [ ]:
# ── 7b. GBM feature importances (load from saved file, or refit if needed) ────
gbm_fi_path = os.path.join(PROJECT_ROOT, "data/outputs/eval/gbm_feature_importances.csv")
fi_df = None

if os.path.exists(gbm_fi_path):
    fi_df = pd.read_csv(gbm_fi_path)
    print(f"Loaded GBM feature importances from {gbm_fi_path}")
else:
    # Refit a GBM to extract importances (run python main.py once to cache this)
    try:
        from src.models.gbm_model import _get_gbm_model, _log_feature_importance
        gbm_model_obj = _get_gbm_model(cfg)
        gbm_model_obj.fit(X_train, y_train)
        fi_df = _log_feature_importance(gbm_model_obj, feature_cols)
        print("Refitted GBM for feature importances (run 'python main.py' to cache this).")
    except Exception as e:
        fi_df = None
        print(f"⚠ GBM not available: {e}")

if fi_df is not None:
    fig, ax = plt.subplots(figsize=(9, 5))
    fi_sorted = fi_df.sort_values("importance", ascending=True)
    ax.barh(fi_sorted["feature"], fi_sorted["importance"],
            color=METHOD_COLORS["gbm"], edgecolor="white", linewidth=0.5, alpha=0.85)
    ax.set_xlabel("Feature Importance (gain)")
    ax.set_title("Figure 11 — GBM Feature Importances\n(gain = average improvement per split using that feature)",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print("\nGBM feature importances:")
    print(fi_df.sort_values("importance", ascending=False).to_string(index=False))
else:
    print("GBM feature importances not available.")

---
## Section 8 — Evaluation Summary

Comparative metrics across all methods. **Read with the key caveat**: no fine-resolution ground truth exists — all metrics measure proxy agreement, not real reconstruction accuracy.

In [ ]:
# ── 8a. Evaluation summary table ─────────────────────────────────────────────
display_cols = [c for c in ["admin_mae_mean_pp", "pearson_r_vs_neg_rwi", "spearman_r_vs_neg_rwi",
                             "ci_mean_width", "ci_median_width"] if c in eval_summary.columns]
summary_display = eval_summary[display_cols].copy() if display_cols else eval_summary.copy()

rename_map = {
    "admin_mae_mean_pp": "Admin MAE (pp)",
    "pearson_r_vs_neg_rwi": "Pearson r (vs −RWI)",
    "spearman_r_vs_neg_rwi": "Spearman r (vs −RWI)",
    "ci_mean_width": "Mean CI Width (pp)",
    "ci_median_width": "Median CI Width (pp)",
}
summary_display = summary_display.rename(columns=rename_map)
summary_display.index.name = "Method"

print("Evaluation Summary — Jamaica 2022, Moderate Poverty")
print("=" * 65)
print(summary_display.round(4).to_string())
print()
print("⚠ IMPORTANT CAVEATS:")
print("  1. Admin MAE ≈ 0 for ALL methods (by construction — hard reconciliation).")
print("  2. Pearson/Spearman r measures agreement with −RWI, not ground truth accuracy.")
print("  3. RWI redistribution has highest r(−RWI) because it IS a function of RWI.")
print("  4. No fine-resolution ground truth exists → absolute accuracy is unknowable.")

In [ ]:
# ── 8b. Visual metric comparison bar chart ────────────────────────────────────
metric_pairs = [
    ("pearson_r_vs_neg_rwi",  "Pearson r (vs −RWI)"),
    ("spearman_r_vs_neg_rwi", "Spearman r (vs −RWI)"),
]
metric_pairs = [(k, v) for k, v in metric_pairs if k in eval_summary.columns]

if metric_pairs:
    fig, axes = plt.subplots(1, len(metric_pairs), figsize=(5 * len(metric_pairs), 4.5))
    if len(metric_pairs) == 1:
        axes = [axes]

    for ax, (metric, label) in zip(axes, metric_pairs):
        methods_idx = eval_summary.index.tolist()
        vals = eval_summary[metric].values
        bar_colors = [method_color_map.get(m.capitalize(), "#888888") for m in methods_idx]
        bars = ax.bar(methods_idx, vals, color=bar_colors, edgecolor="white", width=0.55)
        for bar, val in zip(bars, vals):
            if not np.isnan(val):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                        f"{val:.3f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
        ax.set_ylim(0, 1.1)
        ax.set_ylabel(label)
        ax.set_title(label)
        ax.set_xticklabels(methods_idx, rotation=10, ha="right")

    plt.suptitle("Figure 12 — Proxy Agreement Metrics (higher = more agreement with RWI)\n"
                 "⚠ Circular for RWI method — not evidence of better ground-truth accuracy",
                 fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

---
## Section 9 — Key Findings & Limitations

### What this pipeline achieves

1. **Hard reconciliation** — all methods preserve official zone totals to floating-point precision (diff < 1e-8 pp).
2. **Spatial disaggregation** — RWI redistribution, Ridge, and GBM each produce smooth within-zone gradients that vary from the zone mean, providing finer spatial resolution than the uniform baseline.
3. **Interpretability** — Ridge coefficients clearly show which features drive deprivation estimates; the pipeline is fully reproducible and modular.
4. **Uncertainty quantification** — Bootstrap confidence intervals flag regions where predictions are less stable.

### What this pipeline cannot claim

1. **No fine-resolution ground truth** — Jamaica has no parish-level or grid-level poverty census. All spatial variation within zones is inferred from proxy signals alone, not observed poverty.
2. **Only 3 supervision signals** — with Urban/Rural/KMA as the only target zones, models cannot learn more spatial variation than these 3 values encode. ML models in this regime do not meaningfully outperform RWI redistribution.
3. **Proxy ≠ truth** — RWI correlation measures proxy agreement, not reconstruction accuracy. Reporting "r = 0.93 with RWI" for the RWI method is circular.
4. **Outputs are NOT official statistics** — they represent one plausible spatial disaggregation consistent with official zone totals.

### Recommended next steps

- Obtain parish-level poverty data (14 parishes) to enable stronger evaluation and more constrained ML training.
- Add SHAP values for cell-level interpretability.
- Extend to other MICS countries to test cross-country generalization.
- Add night-time lights and distance-to-services as additional proxy features.